In [1]:
import requests
import tkinter as tk
from tkinter import messagebox
import matplotlib.pyplot as plt


plt.rcParams['font.family'] = 'Arial'


# API CONFIG 
BOOK_SEARCH_URL = "https://openlibrary.org/search.json"
AUTHOR_SEARCH_URL = "https://openlibrary.org/search/authors.json"


# API CALLS
def search_books_api(keyword: str):
    params = {"q": keyword}
    response = requests.get(BOOK_SEARCH_URL, params=params)

    if response.status_code == 200:
        return response.json()
    else:
        return None


# DATA PROCESSING
def get_book_info(book):
    title = book.get("title", "Unknown Title")
    authors = ", ".join(book.get("author_name", ["Unknown Author"]))
    year = book.get("first_publish_year", "N/A")
    return title, authors, year


# GUI
class LibraryApp:
    def __init__(self, master):
        self.master = master
        master.title("OPENLIBRARY SEARCH TOOL")

        # Input
        self.query_label = tk.Label(master, text="Search Keyword:")
        self.query_label.grid(row=0, column=0, padx=5, pady=5, sticky="e")
        
        self.query_entry = tk.Entry(master, width=35)
        self.query_entry.grid(row=0, column=1, padx=5, pady=5)

        # buttons
        self.book_button = tk.Button(master, text="Search Books", command=self.search_books)
        self.book_button.grid(row=0, column=2, padx=0, pady=0)

        # self.author_button = tk.Button(master, text="Search Authors", command=self.search_authors)
        # self.author_button.grid(row=0, column=3, padx=5, pady=5)

        # Output Box
        self.output_box = tk.Text(master, width=80, height=20, wrap="word")
        self.output_box.grid(row=1, column=0, columnspan=4, padx=10, pady=10)
       


         # === BOOK SEARCH ===
    def search_books(self):
        keyword = self.query_entry.get().strip()
        if not keyword:
            messagebox.show_warning("Input Error", "Please enter a keyword.")
            return

        data = search_books_api(keyword)
        if data is None:
            messagebox.show_error("Error", "Could not get book information.")
            return

        results = data.get("docs", [])
        self.output_box.delete("1.0", tk.END)

        if not results:
            self.output_box.insert(tk.END, "No books found.")
            return

        for i, book in enumerate(results[:50], start=1):
            title, authors, year = get_book_info(book)

            self.output_box.insert(tk.END, f"Book {i}\n")
            self.output_box.insert(tk.END, f"Title: {title}\n")
            self.output_box.insert(tk.END, f"Author(s): {authors}\n")
            self.output_box.insert(tk.END, f"Published: {year}\n")
            self.output_box.insert(tk.END, "-" * 70 + "\n\n")
        

In [2]:
#RUN APPLICATION
root = tk.Tk()
app = LibraryApp(root)
root.mainloop()